CEL -> prognozę OZE - watr + słońce - OZE to sobie jeszcze sprawdzimy
ile z wiatru jesteśmy w stanie wygenerować energii na kolejne 24 godziny sądząc po samej prognozie pogody i danych historycznych z 2 lat
ile z słońca jesteśmy w stanie wygenerować energii na kolejne 24 godziny sądząc po samej prognozie pogody i danych historycznych z 2 lat
ale z jednego modelu

jaką granulację prognozy czy co godzinę czy na 24h - co godzinę lepsze
prognoz pogody możemy użyć na kolejny dzień - jeśli chodzi o test.

zakres danych od 2023 do 2025 - na tym trenowanie - modele różne (testujemy jako okno a nie tylko np ostatni miesiąc)


Z danych bierzemy słońce, wiatr, temperaturę oraz wilgotność

z tego https://transparency.entsoe.eu/load/total/dayAhead?appState=%7B"sa"%3A%5B"CTY%7C10YPL-AREA-----S"%5D%2C"st"%3A"CTY"%2C"mm"%3Afalse%2C"ma"%3Afalse%2C"sp"%3A"HALF"%2C"dt"%3A"CHART"%2C"df"%3A%5B"2026-01-01"%2C"2026-01-01"%5D%2C"tz"%3A"CET"%7D - ogarnąć klucz do API



In [ ]:
# Pogoda dla polski:


# pip install openmeteo-requests
# pip install requests-cache retry-requests numpy pandas

import openmeteo_requests

import pandas as pd
import requests_cache
from retry_requests import retry

# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
params = {
	"latitude": 52,
	"longitude": 20,
	"hourly": "temperature_2m",
	"past_days": 0,
	"forecast_days": 7,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()

hourly_data = {"date": pd.date_range(
	start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
	end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
	freq = pd.Timedelta(seconds = hourly.Interval()),
	inclusive = "left"
)}

hourly_data["temperature_2m"] = hourly_temperature_2m

hourly_dataframe = pd.DataFrame(data = hourly_data)
print("\nHourly data\n", hourly_dataframe)